# Multi Query Retriever — 쿼리 다각화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 단일 쿼리 검색의 한계와 **MultiQueryRetriever**가 이를 보완하는 원리 이해하기
- `MultiQueryRetriever.from_llm()`으로 LLM 기반 쿼리 다각화 구현하기
- 로깅으로 LLM이 생성한 쿼리 목록을 직접 확인하기

## 사전 지식

- VectorStoreRetriever 기본 사용법
- LLM이 텍스트를 이해하고 변환하는 원리

---

## 핵심 아이디어

하나의 쿼리를 LLM이 여러 관점으로 재구성해요:

| | 내용 |
|---|---|
| **원본 쿼리** | "딥러닝이 뭐야?" |
| **변환 쿼리 1** | "딥러닝의 정의는 무엇인가?" |
| **변환 쿼리 2** | "딥러닝 기술에 대해 설명해주세요" |
| **변환 쿼리 3** | "딥러닝은 어떤 학습 방법인가?" |

각 쿼리로 검색 후 결과를 통합(중복 제거)합니다.

> **주의**: MultiQueryRetriever는 LLM을 추가로 호출하기 때문에 응답 속도가 느려지고 비용이 증가해요. 검색 품질이 중요한 경우에 선택적으로 사용하세요.

> **실무 팁**: 로깅 레벨을 INFO로 설정하면 LLM이 생성한 쿼리들을 콘솔에서 확인할 수 있어요.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# MultiQueryRetriever를 위한 기본 Retriever와 LLM 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt 기반 FAISS Retriever와 ChatOpenAI LLM을 생성하세요
# 힌트: base_retriever는 k=3, LLM은 temperature=0 (일관된 쿼리 생성)
# 예상 결과: Retriever와 LLM 준비 완료 메시지 출력
# ============================================================

# 문서 및 retriever 준비
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# temperature=0: 쿼리 생성 결과를 일관되게 유지
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

print("✅ 기본 Retriever 및 LLM 준비 완료")

In [ ]:
# ---------------------------------------------------
# MultiQueryRetriever 생성 및 LLM 생성 쿼리 확인
# ---------------------------------------------------

# ============================================================
# TODO: MultiQueryRetriever.from_llm()으로 생성하고 INFO 로그로 생성된 쿼리를 확인하세요
# 힌트: logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)
# 예상 결과: LLM이 생성한 여러 쿼리가 로그에 출력되고 문서가 검색됨
# ============================================================

from langchain.retrievers.multi_query import MultiQueryRetriever
import logging

# 로깅 설정 — INFO 레벨로 LLM이 생성한 쿼리 목록 확인 가능
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# MultiQueryRetriever: 원본 쿼리를 LLM이 3~5개로 재구성


# 검색
query = "NLP에서 임베딩은 어떤 역할을 하나요?"
docs = multi_query_retriever.invoke(query)

print(f"\n📝 원본 쿼리: {query}")
print(f"\n검색된 문서 수: {len(docs)}개\n")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"[문서 {i}]")
    print(doc.page_content[:150] + "...")
    print("-"*80)

print("\n💡 MultiQueryRetriever의 장점:")
print("  - LLM이 자동으로 쿼리 다각화")
print("  - 다양한 표현으로 더 많은 관련 문서 발견")
print("  - 중복 자동 제거")

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | LLM이 원본 쿼리 → 여러 관점의 쿼리를 자동 생성, 결과 합산 |
| 장점 | 단일 쿼리 한계 극복, 다양한 표현으로 recall(재현율) 향상 |
| 단점 | LLM 추가 호출 비용 발생, 응답 지연 증가 |
| 중복 제거 | 내부적으로 중복 문서를 자동으로 제거해줘요 |
| 로깅 확인 | `logging.getLogger("langchain...").setLevel(logging.INFO)` |

### 생성 쿼리 예시

원본 쿼리: "머신러닝에서 과적합을 방지하는 방법은?"

| 생성 쿼리 | 관점 |
|-----------|------|
| 과적합 문제를 해결하기 위한 정규화 기법은? | 기법 중심 |
| 딥러닝 모델의 일반화 성능을 높이는 전략은? | 목적 중심 |
| Train/Test 성능 차이를 줄이는 방법은? | 증상 중심 |

### 다음 노트북 예고

**07-MultiVectorRetriever** — 원본 문서 대신 요약본이나 가상 질문으로 임베딩하여 검색하고, 실제 반환은 원본 문서를 돌려주는 전략을 배워요.